# Credit Risk Scorecard
**Dataset:** Give Me Some Credit (Kaggle)  
**Models:** Logistic Regression Scorecard (WoE/IV) + XGBoost  
**Output:** Probability of Default (PD), Scorecard Points, Expected Loss Metrics

---

## 1. Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.preprocessing import prepare_data, load_data, handle_missing_values, cap_outliers, FEATURES, TARGET
from src.binning import fit_woe_bins, transform_woe, plot_woe_chart
from src.scorecard import fit_scorecard, to_points_table, predict_proba_scorecard, predict_score
from src.xgboost_model import fit_xgboost, predict_proba_xgb, plot_feature_importance, plot_shap_summary
from src.evaluation import evaluation_report, compare_models, plot_roc_curves, plot_ks_chart

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_PATH = '../data/cs-training.csv'
print('Setup complete')

## 2. Load & Explore Data

In [ ]:
df_raw = load_data(DATA_PATH)
print(f'Shape: {df_raw.shape}')
print(f'\nTarget distribution:')
print(df_raw[TARGET].value_counts(normalize=True).rename({0: 'Good (No Default)', 1: 'Bad (Default)'}))
df_raw.head()

In [ ]:
# Missing values
missing = df_raw.isnull().mean().sort_values(ascending=False)
missing = missing[missing > 0]
print('Missing value rates:')
print(missing.apply(lambda x: f'{x:.1%}'))

In [ ]:
# Distribution of key features by target
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
plot_features = ['RevolvingUtilizationOfUnsecuredLines', 'age', 'DebtRatio',
                 'MonthlyIncome', 'NumberOfTimes90DaysLate', 'NumberOfTime30-59DaysPastDueNotWorse']

df_plot = df_raw.copy()
df_plot['MonthlyIncome'] = df_plot['MonthlyIncome'].fillna(df_plot['MonthlyIncome'].median())
df_plot['MonthlyIncome'] = df_plot['MonthlyIncome'].clip(0, df_plot['MonthlyIncome'].quantile(0.99))

for ax, feat in zip(axes.flat, plot_features):
    for label, grp in df_plot.groupby(TARGET):
        grp[feat].clip(grp[feat].quantile(0.01), grp[feat].quantile(0.99)).hist(
            ax=ax, alpha=0.6, bins=40, label='Default' if label == 1 else 'Good', density=True
        )
    ax.set_title(feat, fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Target', y=1.01, fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Preprocessing & Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = prepare_data(DATA_PATH, test_size=0.3, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train default rate: {y_train.mean():.2%}')
print(f'Test default rate:  {y_test.mean():.2%}')

## 4. WoE Binning & Information Value

In [ ]:
binner, iv_table, selected_features = fit_woe_bins(X_train, y_train, min_iv=0.02)
print(f'Features selected (IV >= 0.02): {len(selected_features)}')
iv_table.style.background_gradient(subset=['iv'], cmap='RdYlGn')

In [ ]:
# WoE charts for top 4 features
top_features = iv_table.head(4)['feature'].tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, feat in zip(axes.flat, top_features):
    plot_woe_chart(binner, feat, ax=ax)
plt.suptitle('WoE Profiles — Top 4 Features by IV', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/woe_charts.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Transform to WoE features
X_train_woe = transform_woe(binner, X_train[selected_features])
X_test_woe = transform_woe(binner, X_test[selected_features])
print(f'WoE-transformed shape: {X_train_woe.shape}')

## 5. Logistic Regression Scorecard

In [ ]:
lr_model = fit_scorecard(X_train_woe, y_train)
print('Logistic Regression fitted.')
print(f'Coefficients: {dict(zip(selected_features, lr_model.coef_[0].round(4)))}')

In [ ]:
# Scorecard points table
points_table = to_points_table(
    lr_model, binner, selected_features,
    pdo=20, base_score=600, base_odds=19.0
)
print('Scorecard Points Table (PDO=20, Base Score=600, Base Odds=19:1)')
points_table.style.background_gradient(subset=['points'], cmap='RdYlGn')

In [ ]:
# Save scorecard to Excel
points_table.to_excel('../outputs/scorecard_points.xlsx', index=False)
print('Saved to outputs/scorecard_points.xlsx')

In [ ]:
# Predict PD and scores
lr_prob_train = predict_proba_scorecard(lr_model, X_train_woe)
lr_prob_test = predict_proba_scorecard(lr_model, X_test_woe)

# Score distribution
test_scores = predict_score(lr_model, binner, X_test, selected_features)
fig, ax = plt.subplots(figsize=(8, 4))
for label, color in [(0, '#1a9641'), (1, '#d73027')]:
    mask = y_test == label
    ax.hist(test_scores[mask], bins=40, alpha=0.6,
            label='Good' if label == 0 else 'Default', color=color, density=True)
ax.set_xlabel('Scorecard Points')
ax.set_ylabel('Density')
ax.set_title('Score Distribution by Outcome')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. XGBoost Model

In [ ]:
# Train on raw features (no WoE transformation)
# Use a validation split from training data for early stopping
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.15, stratify=y_train, random_state=42)

xgb_model = fit_xgboost(X_tr, y_tr, X_val, y_val)
print(f'Best iteration: {xgb_model.best_iteration}')

xgb_prob_test = predict_proba_xgb(xgb_model, X_test)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_feature_importance(xgb_model, X_train.columns.tolist(), ax=ax)
plt.savefig('../outputs/xgb_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP summary plot
X_sample = X_test.sample(500, random_state=42)
plot_shap_summary(xgb_model, X_sample)

## 7. Model Evaluation

In [ ]:
report_lr = evaluation_report(y_test, lr_prob_test, 'Logistic Regression Scorecard')
report_xgb = evaluation_report(y_test, xgb_prob_test, 'XGBoost')

results = compare_models([report_lr, report_xgb])
print('Model Comparison on Test Set')
results.style.highlight_max(axis=0, color='#c6efce')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

plot_roc_curves(
    {'Logistic Regression': lr_prob_test, 'XGBoost': xgb_prob_test},
    y_test.values, ax=ax1
)
plot_ks_chart(y_test.values, xgb_prob_test, 'XGBoost', ax=ax2)

plt.savefig('../outputs/roc_ks_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Expected Loss Framework

**Expected Loss (EL) = PD × LGD × EAD**

| Component | Definition | Assumption |
|-----------|-----------|------------|
| **PD** | Probability of Default | From our scorecard model |
| **LGD** | Loss Given Default | 0.45 (Basel II unsecured retail) |
| **EAD** | Exposure at Default | Normalised to 1.0 (unit exposure) |

*Note: The Give Me Some Credit dataset does not contain loan balances, so EAD is treated as a unit value. In production, EAD would be the outstanding loan balance.*

In [ ]:
LGD = 0.45  # Basel II assumption for unsecured retail
EAD = 1.0   # Unit exposure (normalised)

# Use XGBoost PD (typically better calibrated)
pd_values = xgb_prob_test
el_values = pd_values * LGD * EAD

el_df = pd.DataFrame({
    'PD': pd_values,
    'LGD': LGD,
    'EAD': EAD,
    'Expected_Loss': el_values,
    'Actual_Default': y_test.values,
})

print('Portfolio-Level Expected Loss Summary')
print(f'  Mean PD:              {pd_values.mean():.2%}')
print(f'  Mean Expected Loss:   {el_values.mean():.4f}')
print(f'  Total Expected Loss:  {el_values.sum():.2f}  (on {len(el_values):,} accounts, unit EAD)')
print(f'  Actual Default Rate:  {y_test.mean():.2%}')

In [ ]:
# EL distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PD distribution
axes[0].hist(pd_values, bins=60, color='#2166ac', edgecolor='white', alpha=0.8)
axes[0].axvline(pd_values.mean(), color='red', linestyle='--', label=f'Mean PD = {pd_values.mean():.2%}')
axes[0].set_xlabel('Predicted PD')
axes[0].set_ylabel('Count')
axes[0].set_title('Probability of Default Distribution')
axes[0].legend()

# EL by decile
el_df['decile'] = pd.qcut(el_df['PD'], 10, labels=[f'D{i}' for i in range(1, 11)])
decile_stats = el_df.groupby('decile').agg(
    mean_el=('Expected_Loss', 'mean'),
    default_rate=('Actual_Default', 'mean')
).reset_index()

x = range(len(decile_stats))
axes[1].bar(x, decile_stats['mean_el'], color='#d73027', alpha=0.7, label='Mean EL')
ax_r = axes[1].twinx()
ax_r.plot(x, decile_stats['default_rate'], 'o-', color='#1a9641', lw=2, label='Actual Default Rate')
ax_r.set_ylabel('Actual Default Rate', color='#1a9641')
axes[1].set_xticks(x)
axes[1].set_xticklabels(decile_stats['decile'])
axes[1].set_xlabel('Risk Decile (D1=Lowest Risk)')
axes[1].set_ylabel('Mean Expected Loss')
axes[1].set_title('Expected Loss vs Actual Default Rate by Decile')
axes[1].legend(loc='upper left')
ax_r.legend(loc='upper center')

plt.tight_layout()
plt.savefig('../outputs/expected_loss.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Risk segmentation by score band
el_df['score'] = test_scores
bins = [0, 550, 600, 650, 700, 999]
labels = ['<550 (High Risk)', '550–600', '600–650', '650–700', '>700 (Low Risk)']
el_df['score_band'] = pd.cut(el_df['score'], bins=bins, labels=labels)

band_stats = el_df.groupby('score_band').agg(
    count=('PD', 'count'),
    mean_pd=('PD', 'mean'),
    mean_el=('Expected_Loss', 'mean'),
    actual_dr=('Actual_Default', 'mean')
).round(4)

print('Risk Segmentation by Score Band')
band_stats

## 9. Conclusions

### Model Performance Summary

| Metric | Logistic Regression | XGBoost |
|--------|--------------------|---------|
| AUC-ROC | see above | see above |
| KS Statistic | see above | see above |
| Gini Coefficient | see above | see above |

### Key Takeaways

- **XGBoost** generally achieves higher AUC/KS due to its ability to capture non-linear relationships and interactions between features, without feature engineering.
- **Logistic Regression Scorecard** is more interpretable and regulatory-friendly. It produces integer scores that can be explained to applicants and audited by regulators (e.g., ECOA adverse action reasons).
- The **WoE transformation** converts non-linear relationships into linear ones, enabling logistic regression to compete well with more complex models.
- **Expected Loss** segmentation confirms the model discriminates effectively — higher deciles carry meaningfully higher actual default rates.

### Next Steps

- Calibrate PD outputs using Platt scaling or isotonic regression
- Add LightGBM as a third model for comparison
- Implement reject inference to address sampling bias
- Add MLflow experiment tracking for reproducibility
- Stress-test EL under different LGD scenarios (0.3, 0.45, 0.6)